# 2장 2강: 분산분석(ANOVA)과 사후 검정 이론 — 실습문제

## 실습 목표

- 세 집단 이상의 평균을 일원배치 분산분석으로 비교할 수 있다.
- F통계량과 p-value를 이용하여 전체 집단 차이를 판단할 수 있다.
- 집단 간·집단 내 변동으로 ANOVA 표를 구성하고 해석할 수 있다.
- ANOVA가 유의할 때 Tukey HSD 사후 검정을 수행할 수 있다.
- 유의한 집단 쌍과 평균 차이의 크기를 근거로 차이 구조를 설명할 수 있다.

## 실습 환경 / 데이터

- Python
- NumPy, pandas
- scipy.stats
- statsmodels
- `ames_housing(1).csv`

| 컬럼 | 의미 |
|---|---|
| `SalePrice` | 주택 판매가격 |
| `OverallQual` | 주택의 전반적인 품질 점수 |
| `KitchenQual` | 주방 품질 |

> 모든 검정의 유의수준은 `α = 0.05`입니다.  
> 표본은 지정된 `random_state`로 추출하여 결과를 재현합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. Ames Housing 데이터를 `df`에 불러오세요.
3. 데이터 크기, 결측치 수, 컬럼명과 상위 5개 행을 확인하세요.

In [2]:
# 실습 준비 코드를 작성하세요.
import pandas as pd
from scipy import stats

df = pd.read_csv('ames_housing.csv')

df.describe()

df.head()

,SalePrice,GrLivArea,LotArea,OverallQual,KitchenQual,CentralAir,HeatingQC,PavedDrive,Neighborhood,YearBuilt
0,208500,1710,8450,7,Gd,Y,Ex,Y,CollgCr,2003
1,181500,1262,9600,6,TA,Y,Ex,Y,Veenker,1976
2,223500,1786,11250,7,Gd,Y,Ex,Y,CollgCr,2001
3,140000,1717,9550,7,Gd,Y,Gd,Y,Crawfor,1915
4,250000,2198,14260,8,Gd,Y,Ex,Y,NoRidge,2000


In [1]:
# ANOVA(일원 분산 분석) : 하나의 집단 구분 기준에 따라 나뉜 여러 집단의 모평균이 모두 같은지 검정하는 방법
# -> 교육 방법 A, B, C로 교육을 진행한 뒤 점수를 비교한다.

# F 통계량 : 집단 간 평균제곱을 집단 내 평균제곱으로 나눈 값.
# -> 귀무가설과 가정이 맞을 때, 두 평균제곱은 같은 분산을 추정한다.
# -> F값이 클수록 집단 간 차이가 내부 퍼짐에 비해 크다

---

## 필수 1. One-way ANOVA로 전체 평균 차이 확인

### 문제 1-1. 전반적인 품질 점수 5·6·7 집단 비교

#### 문제 설명

주택의 전반적인 품질 점수가 5점, 6점, 7점인 세 집단의 평균 판매가격을 비교하려고 합니다. 각 집단에서 20개씩 표본을 추출하고 일원배치 분산분석을 수행하세요.

#### 요구사항

1. `OverallQual`이 5, 6, 7인 각 집단의 `SalePrice`에서 `n=20`, `random_state=5`로 표본을 추출하세요.
2. 각 집단의 표본 수, 평균, 표준편차를 출력하세요.
3. 세 집단이 서로 다른 주택으로 구성된 독립집단임을 설명하세요.
4. 각 집단에 Shapiro-Wilk 정규성 검정을 수행하세요.
5. 세 집단에 Levene 등분산 검정을 수행하세요.
6. 다음 가설을 작성하세요.
   - H₀: 세 집단의 모집단 평균 판매가격은 모두 같다.
   - H₁: 적어도 한 집단의 모집단 평균 판매가격은 다르다.
7. `stats.f_oneway()`로 일원배치 ANOVA를 수행하세요.
8. F통계량과 p-value를 출력하고 전체 차이 유무를 판단하세요.
9. ANOVA 결과만으로 어느 집단끼리 다른지 알 수 있는지 설명하세요.

#### 해석 질문

**Q1.** 세 집단을 각각 t검정으로 반복 비교하면 어떤 문제가 발생하나요?  
**Q2.** F통계량은 어떤 두 변동의 비율인가요?  
**Q3.** ANOVA 결과 세 집단의 평균 판매가격에는 전체적으로 유의한 차이가 있나요?  
**Q4.** ANOVA 결과만으로 5점·6점·7점 중 어느 집단 쌍이 다른지 알 수 있나요?

#### 제출 결과

- 집단별 기술통계량과 가정 점검 결과
- 가설 설정
- F통계량과 p-value
- 전체 차이 판단
- 사후 검정 필요성 설명
- Q1~Q4 답변

In [3]:
# 필수 1 코드를 작성하세요.
import numpy as np
import pandas as pd
from scipy import stats

# 1. OverallQual이 5, 6, 7인 집단에서 n=20, random_state=5 표본 추출
sample_q5 = (
    df[df["OverallQual"] == 5]["SalePrice"].dropna().sample(n=20, random_state=5)
)
sample_q6 = (
    df[df["OverallQual"] == 6]["SalePrice"].dropna().sample(n=20, random_state=5)
)
sample_q7 = (
    df[df["OverallQual"] == 7]["SalePrice"].dropna().sample(n=20, random_state=5)
)

groups = {"품질 5점": sample_q5, "품질 6점": sample_q6, "품질 7점": sample_q7}

# 2. 각 집단의 표본 수, 표본평균, 표본표준편차 출력
print("=== 1. 기술통계량 ===")
for name, data in groups.items():
    print(
        f"{name}: 표본 수 = {len(data)}, 평균 = ${data.mean():,.2f}, 표준편차 = ${data.std(ddof=1):,.2f}"
    )

# 3. 독립성 설명
print("\n=== 2. 독립성 확인 ===")
print(
    "설명: 각 집단은 완전히 분리된 개별 주택들로 구성되어 있으며, 상호 간에 영향을 미치지 않는 독립표본입니다."
)

# 4. Shapiro-Wilk 정규성 검정
print("\n=== 3. 정규성 검정 (Shapiro-Wilk) ===")
for name, data in groups.items():
    stat, p = stats.shapiro(data)
    print(f"{name}: W-통계량 = {stat:.4f}, p-value = {p:.4f}")

# 5. Levene 등분산 검정
stat_lev, p_lev = stats.levene(sample_q5, sample_q6, sample_q7)
print("\n=== 4. 등분산성 검정 (Levene) ===")
print(
    f"통계량 = {stat_lev:.4f}, p-value = {p_lev:.4f} (등분산 충족 여부: {p_lev >= 0.05})"
)

# 7. stats.f_oneway()로 일원배치 ANOVA 수행
f_stat, p_val = stats.f_oneway(sample_q5, sample_q6, sample_q7)

# 8. F통계량과 p-value 출력 및 유의성 판정
print("\n=== 5. One-way ANOVA 결과 ===")
print(f"F-통계량: {f_stat:.4f}")
print(f"p-value : {p_val:.4e}")

alpha = 0.05
if p_val < alpha:
    print(
        f"판정: p-value({p_val:.4e}) < {alpha} 이므로 귀무가설(H₀)을 기각합니다."
    )
    print("      적어도 한 집단의 평균 판매가격은 통계적으로 다릅니다.")
else:
    print(
        f"판정: p-value({p_val:.4e}) >= {alpha} 이므로 귀무가설(H₀)을 기각하지 못합니다."
    )

=== 1. 기술통계량 ===
품질 5점: 표본 수 = 20, 평균 = $130,605.00, 표준편차 = $24,937.11
품질 6점: 표본 수 = 20, 평균 = $167,826.60, 표준편차 = $41,944.55
품질 7점: 표본 수 = 20, 평균 = $217,593.60, 표준편차 = $48,298.39

=== 2. 독립성 확인 ===
설명: 각 집단은 완전히 분리된 개별 주택들로 구성되어 있으며, 상호 간에 영향을 미치지 않는 독립표본입니다.

=== 3. 정규성 검정 (Shapiro-Wilk) ===
품질 5점: W-통계량 = 0.9710, p-value = 0.7760
품질 6점: W-통계량 = 0.9527, p-value = 0.4096
품질 7점: W-통계량 = 0.9259, p-value = 0.1290

=== 4. 등분산성 검정 (Levene) ===
통계량 = 2.6516, p-value = 0.0792 (등분산 충족 여부: True)

=== 5. One-way ANOVA 결과 ===
F-통계량: 24.2456
p-value : 2.4031e-08
판정: p-value(2.4031e-08) < 0.05 이므로 귀무가설(H₀)을 기각합니다.
      적어도 한 집단의 평균 판매가격은 통계적으로 다릅니다.


### 필수 1 답변 작성란

**Q1.** 세 집단을 각각 t검정으로 반복 비교하면 어떤 문제가 발생하나요?  
- 검정을 반복할수록 전체 분석에서 한 번 이상 제 1종 오류가 발생할 확률이 0.05보다 커지는 다중 비교 문제가 발생한다.

**Q2.** F통계량은 어떤 두 변동의 비율인가요?  
- 집단 간 평균 차이를 나타내는 집단 간 변동을 같은 집단 내부의 개인차인 집단 내 변동으로 나눈 비율

**Q3.** ANOVA 결과 세 집단의 평균 판매가격에는 전체적으로 유의한 차이가 있나요?  
- 있다. 따라서 적어도 한 집단의 모집단 평균 판매가격은 다르다.

**Q4.** ANOVA 결과만으로 5점·6점·7점 중 어느 집단 쌍이 다른지 알 수 있나요?
-  알 수 없음, 구체적인 집단 쌍은 Tukey HSD와 같은 사후 검정으로 확인해야한다.

---

## 필수 2. ANOVA 표 구성과 Tukey HSD 사후 검정

### 문제 2-1. 품질 점수별 차이 구조 확인

#### 문제 설명

필수 1의 세 집단을 이용하여 ANOVA 표를 직접 구성하고, 전체 차이가 유의한 경우 Tukey HSD 사후 검정을 수행해 어느 품질 점수 집단끼리 차이가 있는지 확인하세요.

#### 요구사항

1. 필수 1의 세 집단을 하나의 `anova_df` 데이터프레임으로 결합하세요.
2. 전체 평균을 계산하세요.
3. 다음 값을 계산하여 ANOVA 표를 만드세요.
   - 집단 간 제곱합 `SS_between`
   - 집단 내 제곱합 `SS_within`
   - 집단 간·집단 내 자유도
   - 평균제곱 `MS_between`, `MS_within`
   - F통계량
4. 직접 계산한 F통계량이 `stats.f_oneway()` 결과와 일치하는지 확인하세요.
5. ANOVA p-value가 0.05보다 작을 때만 `pairwise_tukeyhsd()`를 실행하세요.
6. Tukey 결과에서 `reject=True`인 집단 쌍을 확인하세요.
7. 각 집단 평균을 이용하여 집단 쌍별 평균 차이를 계산하세요.
8. 어느 집단 쌍이 유의하며 차이가 가장 큰 집단 쌍은 무엇인지 해석하세요.

#### 해석 질문

**Q1.** ANOVA 표에서 `SS_between`과 `SS_within`은 각각 무엇을 의미하나요?  
**Q2.** Tukey HSD의 `reject=True`는 무엇을 의미하나요?  
**Q3.** 어느 품질 점수 집단 쌍에서 유의한 차이가 확인되나요?  
**Q4.** 평균 판매가격 차이가 가장 큰 집단 쌍은 무엇이며 차이는 얼마인가요?

#### 제출 결과

- ANOVA 표
- F통계량 대조 결과
- Tukey HSD 결과
- 유의한 집단 쌍
- 집단 쌍별 평균 차이
- Q1~Q4 답변

In [4]:
# 필수 2 코드를 작성하세요.
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# 1. 필수 1 데이터 결합 (n=20, random_state=5 표본)
sample_q5 = (
    df[df["OverallQual"] == 5]["SalePrice"].dropna().sample(n=20, random_state=5)
)
sample_q6 = (
    df[df["OverallQual"] == 6]["SalePrice"].dropna().sample(n=20, random_state=5)
)
sample_q7 = (
    df[df["OverallQual"] == 7]["SalePrice"].dropna().sample(n=20, random_state=5)
)

anova_df = pd.DataFrame(
    {
        "SalePrice": pd.concat(
            [sample_q5, sample_q6, sample_q7], ignore_index=True
        ),
        "OverallQual": ["5"] * 20 + ["6"] * 20 + ["7"] * 20,
    }
)

# 2. 전체 평균 및 집단별 기초 통계
grand_mean = anova_df["SalePrice"].mean()
group_stats = anova_df.groupby("OverallQual")["SalePrice"].agg(
    ["count", "mean", "var"]
)

# 3. ANOVA 표 구성 요소 직접 계산
k = len(group_stats)  # 집단 수 (3)
N = len(anova_df)  # 전체 데이터 수 (60)

# 집단 간 제곱합 (SS_between) 및 자유도 (df_between)
ss_between = sum(
    group_stats["count"] * ((group_stats["mean"] - grand_mean) ** 2)
)
df_between = k - 1

# 집단 내 제곱합 (SS_within) 및 자유도 (df_within)
ss_within = sum((group_stats["count"] - 1) * group_stats["var"])
df_within = N - k

# 총 제곱합 (SS_total) 및 자유도
ss_total = ss_between + ss_within
df_total = N - 1

# 평균제곱 (MS)
ms_between = ss_between / df_between
ms_within = ss_within / df_within

# F통계량 및 p-value
f_manual = ms_between / ms_within
p_manual = stats.f.sf(f_manual, df_between, df_within)

anova_table = pd.DataFrame(
    {
        "요인": ["집단 간 (Between)", "집단 내 (Within)", "총합 (Total)"],
        "제곱합 (SS)": [ss_between, ss_within, ss_total],
        "자유도 (df)": [df_between, df_within, df_total],
        "평균제곱 (MS)": [ms_between, ms_within, np.nan],
        "F-통계량": [f_manual, np.nan, np.nan],
        "p-value": [p_manual, np.nan, np.nan],
    }
)

print("=== 1. 직접 계산한 ANOVA 표 ===")
print(anova_table.to_string(index=False))

# 4. stats.f_oneway() 결과와 일치 여부 대조
f_stat_func, p_val_func = stats.f_oneway(sample_q5, sample_q6, sample_q7)
print(f"\n직접 계산 F통계량 : {f_manual:.6f}")
print(f"함수 반환 F통계량 : {f_stat_func:.6f}")
print(f"F통계량 일치 여부 : {np.isclose(f_manual, f_stat_func)}")

# 5. ANOVA 유의할 때 Tukey HSD 사후 검정 수행
if p_val_func < 0.05:
    print("\n=== 2. Tukey HSD 사후 검정 결과 ===")
    tukey = pairwise_tukeyhsd(
        endog=anova_df["SalePrice"],
        groups=anova_df["OverallQual"],
        alpha=0.05,
    )
    print(tukey)

# 7. 집단 쌍별 평균 차이 수치
means = {
    "5": sample_q5.mean(),
    "6": sample_q6.mean(),
    "7": sample_q7.mean(),
}
diff_6_5 = means["6"] - means["5"]
diff_7_5 = means["7"] - means["5"]
diff_7_6 = means["7"] - means["6"]

print("\n=== 3. 집단 쌍별 평균 차이 ===")
print(
    f"품질 6점 - 품질 5점 : ${diff_6_5:,.2f} (6점: ${means['6']:,.2f}, 5점: ${means['5']:,.2f})"
)
print(
    f"품질 7점 - 품질 5점 : ${diff_7_5:,.2f} (7점: ${means['7']:,.2f}, 5점: ${means['5']:,.2f})"
)
print(
    f"품질 7점 - 품질 6점 : ${diff_7_6:,.2f} (7점: ${means['7']:,.2f}, 6점: ${means['6']:,.2f})"
)

=== 1. 직접 계산한 ANOVA 표 ===
            요인     제곱합 (SS)  자유도 (df)    평균제곱 (MS)     F-통계량      p-value
집단 간 (Between) 7.619479e+10         2 3.809739e+10 24.245575 2.403126e-08
 집단 내 (Within) 8.956486e+10        57 1.571313e+09       NaN          NaN
    총합 (Total) 1.657596e+11        59          NaN       NaN          NaN

직접 계산 F통계량 : 24.245575
함수 반환 F통계량 : 24.245575
F통계량 일치 여부 : True

=== 2. Tukey HSD 사후 검정 결과 ===
    Multiple Comparison of Means - Tukey HSD, FWER=0.05    
group1 group2 meandiff p-adj    lower       upper    reject
-----------------------------------------------------------
     5      6  37221.6  0.012  7056.6586  67386.5414   True
     5      7  86988.6    0.0 56823.6586 117153.5414   True
     6      7  49767.0 0.0006 19602.0586  79931.9414   True
-----------------------------------------------------------

=== 3. 집단 쌍별 평균 차이 ===
품질 6점 - 품질 5점 : $37,221.60 (6점: $167,826.60, 5점: $130,605.00)
품질 7점 - 품질 5점 : $86,988.60 (7점: $217,593.60, 5점: $130,605.00)
품질 7점 - 품질 6점 

### 필수 2 답변 작성란

**Q1.** ANOVA 표에서 `SS_between`과 `SS_within`은 각각 무엇을 의미하나요?  
- SS_between은 집단 평균들이 전체 평균에서 벗어난 집단 간 변동 값
- SS_within은 각 관측값이 소속 집단 평균에서 벗어난 집단 내 변동

**Q2.** Tukey HSD의 `reject=True`는 무엇을 의미하나요?  
- 다중비교 오류를 조정한 뒤에도 해당 두 집단의 평균이 같다는 귀무가설을 기각하겠다는 의미

**Q3.** 어느 품질 점수 집단 쌍에서 유의한 차이가 확인되나요?  
- 5-6, 5-7, 6-7의 모든 집단 쌍에서 유의한 차이가 확인된다.

**Q4.** 평균 판매가격 차이가 가장 큰 집단 쌍은 무엇이며 차이는 얼마인가요?
- 품질 5점과 7점 집단이며 평균 차이는 약 86,988달러이다.

---

## 과제. 주방 품질에 따른 판매가격 차이 분석

### 문제 3-1. 주방 품질 `Ex`·`Gd`·`TA` 집단 비교

#### 문제 설명

주방 품질이 `Ex`, `Gd`, `TA`인 세 집단의 평균 판매가격을 비교합니다. 각 집단에서 20개씩 표본을 추출한 뒤 ANOVA와 Tukey HSD를 순서대로 적용하세요.

> 필수 문제에서 학습한 전체 검정→사후 검정 절차를 새로운 집단 변수에 적용하는 과제입니다.

#### 요구사항

1. `KitchenQual`이 `Ex`, `Gd`, `TA`인 각 집단의 `SalePrice`에서 `n=20`, `random_state=18`로 표본을 추출하세요.
2. 세 집단의 표본 수와 평균을 출력하세요.
3. 정규성과 등분산성을 확인하세요.
4. 일원배치 ANOVA를 수행하고 F통계량과 p-value를 출력하세요.
5. ANOVA가 유의한 경우에만 Tukey HSD 사후 검정을 수행하세요.
6. Tukey 결과에서 유의한 집단 쌍을 확인하세요.
7. 각 집단 쌍의 평균 판매가격 차이를 계산하세요.
8. 어느 집단 쌍의 차이가 가장 큰지 포함하여 주방 품질별 차이 구조를 해석하세요.

#### 해석 질문

**Q1.** ANOVA 결과 세 집단의 평균에는 전체적으로 유의한 차이가 있나요?  
**Q2.** 사후 검정은 어떤 조건에서 수행하나요?  
**Q3.** Tukey HSD에서 유의한 차이가 확인된 집단 쌍은 무엇인가요?  
**Q4.** 평균 판매가격 차이가 가장 큰 집단 쌍은 무엇이며 차이는 얼마인가요?

#### 제출 결과

- 집단별 기술통계량과 가정 점검
- ANOVA 결과
- Tukey HSD 결과
- 유의한 집단 쌍과 평균 차이
- 최종 해석
- Q1~Q4 답변

In [ ]:
# 과제 코드를 작성하세요.

### 과제 답변 작성란

- **Q1.**
- **Q2.**
- **Q3.**
- **Q4.**

## 실습 마무리

1. 세 집단 이상을 t검정으로 반복 비교하면 왜 제1종 오류가 커지나요?
- 각 검정마다 위양성 가능성이 있기 때문에 비교횟수가 늘어날수록 전체 분석에서 한 번 이상 잘못 기각할 확률이 누적된다.

2. ANOVA의 귀무가설과 대립가설은 무엇인가요?
- 귀무가설은 모든 집단의 모집단과 평균이 같다, 대립가설은 적어도 한 집단의 평균이 다르다라는 것

3. F통계량이 크다는 것은 무엇을 의미하나요?
- 집단 내 변동에 비해 집단 간 평균 차이로 설명되는 변동이 상대적으로 크다는 의미

4. ANOVA가 유의하더라도 사후 검정이 필요한 이유는 무엇인가요?
- ANOVA는 적어도 한 집단이 다르다는 사실만 알려주며 구체적인 집단 쌍은 알려주지 않음

5. Tukey HSD 결과에서 어떤 항목을 확인해야 하나요?
- 비교한 집단 쌍, 평균 차이, 조정돤 P-Value, 신뢰구간, reject 여부